In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
from sqlalchemy import create_engine, URL, text
#import psycopg2

from getpass import getpass
from urllib.parse import quote_plus

In [2]:
data_path = Path("data")
facebook = pd.read_csv(data_path / "01_facebook_ads.csv")
google = pd.read_csv(data_path / "02_google_ads.csv")
tiktok = pd.read_csv(data_path / "03_tiktok_ads.csv")

In [3]:
for df in [facebook, google, tiktok]:
    print(',\n'.join(df.columns))
    print("\n")

date,
campaign_id,
campaign_name,
ad_set_id,
ad_set_name,
impressions,
clicks,
spend,
conversions,
video_views,
engagement_rate,
reach,
frequency


date,
campaign_id,
campaign_name,
ad_group_id,
ad_group_name,
impressions,
clicks,
cost,
conversions,
conversion_value,
ctr,
avg_cpc,
quality_score,
search_impression_share


date,
campaign_id,
campaign_name,
adgroup_id,
adgroup_name,
impressions,
clicks,
cost,
conversions,
video_views,
video_watch_25,
video_watch_50,
video_watch_75,
video_watch_100,
likes,
shares,
comments




In [4]:
facebook_rename = {
    "ad_set_id": "ad_group_id",
    "ad_set_name": "ad_group_name",
}

google_rename = {
    "cost": "spend",
    "conversion_value": "revenue",
    "avg_cpc": "cpc",
}

tiktok_rename = {
    "adgroup_id": "ad_group_id",
    "adgroup_name": "ad_group_name",
    "cost": "spend",
}

facebook.rename(columns=facebook_rename, inplace=True)
google.rename(columns=google_rename, inplace=True)
tiktok.rename(columns=tiktok_rename, inplace=True)

In [5]:
union = pd.concat([
    facebook,
    google,
    tiktok
], 
                  axis=0,
                  names=["platform"],
                  keys=[
                    "facebook",    
                    "google",
                    "tiktok"
                  ]
                 ).reset_index(drop=False, level=[0])
union

,platform,date,campaign_id,campaign_name,ad_group_id,ad_group_name,impressions,clicks,spend,conversions,...,cpc,quality_score,search_impression_share,video_watch_25,video_watch_50,video_watch_75,video_watch_100,likes,shares,comments
0,facebook,2024-01-01,fb_1001,Brand_Awareness_Q1,fbset_2001,Broad_Audience_18-35,45623,892,127.5,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,facebook,2024-01-01,fb_1002,Conversions_Retargeting,fbset_2002,Cart_Abandoners,12456,567,215.8,34,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,facebook,2024-01-01,fb_1003,Traffic_Drive_Jan,fbset_2003,Lookalike_Purchasers,34567,1234,189.9,23,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,facebook,2024-01-02,fb_1001,Brand_Awareness_Q1,fbset_2001,Broad_Audience_18-35,42345,823,118.2,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,facebook,2024-01-02,fb_1002,Conversions_Retargeting,fbset_2002,Cart_Abandoners,13678,623,234.5,38,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105,tiktok,2024-01-29,tt_8004,Influencer_Collab,ttad_9004,Creator_Partnership,423456,6890,1056.8,107,...,NaN,NaN,NaN,289234.0,231456.0,164567.0,109234.0,18890.0,5456.0,1489.0
106,tiktok,2024-01-30,tt_8001,Awareness_GenZ,ttad_9001,Dance_Challenge,295678,4345,573.2,44,...,NaN,NaN,NaN,188234.0,117890.0,79234.0,52789.0,10567.0,2856.0,667.0
107,tiktok,2024-01-30,tt_8002,Conversion_Focus,ttad_9002,Product_Demo,161234,3089,823.7,83,...,NaN,NaN,NaN,83567.0,56234.0,42345.0,28234.0,5523.0,1589.0,301.0
108,tiktok,2024-01-30,tt_8003,Traffic_Campaign,ttad_9003,Trending_Sounds,251234,3967,481.5,35,...,NaN,NaN,NaN,150567.0,119234.0,76890.0,46567.0,8678.0,2378.0,556.0


In [6]:
union.columns

Index(['platform', 'date', 'campaign_id', 'campaign_name', 'ad_group_id',
       'ad_group_name', 'impressions', 'clicks', 'spend', 'conversions',
       'video_views', 'engagement_rate', 'reach', 'frequency', 'revenue',
       'ctr', 'cpc', 'quality_score', 'search_impression_share',
       'video_watch_25', 'video_watch_50', 'video_watch_75', 'video_watch_100',
       'likes', 'shares', 'comments'],
      dtype='str')

In [7]:
password = getpass("DB password: ")
DB_URL = URL.create(
    drivername='postgresql+psycopg',
    username="postgres.oninlvungqpvughmfeum",
    password=password,
    host="aws-0-eu-west-1.pooler.supabase.com",
    port=5432,
    database='postgres'
)
engine = create_engine(DB_URL)

DB password:  ········


In [8]:
query = """
drop table if exists public.facebook;
drop table if exists public.google;
drop table if exists public.tiktok;
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [9]:
tables = {
    "facebook": facebook,
    "google": google,
    "tiktok": tiktok,
    "ads_all": union
}

for table_name, df in tables.items():
    df.to_sql(
        table_name,
        engine,
        schema="public",
        if_exists="replace",   # use "append" if table already exists
        index=False
    )